In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
import joblib

df = pd.read_csv('../data/astana_ground_truth.csv')
df.head()

,pm2_5,heating_degree_days,vehicles_per_minute
0,8.8,7.7,0.000000
1,10.9,7.1,23.529801
2,17.1,6.8,24.110418
3,16.4,5.9,31.010303
4,16.5,4.8,23.907758


In [2]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df.values)
joblib.dump(scaler, '../data/scaler.pkl')

def create_sequences(data, seq_len=24, predict_ahead=24):
    X, y = [], []
    for i in range(len(data) - seq_len - predict_ahead + 1):
        X.append(data[i:(i + seq_len)])
        y.append(data[(i + seq_len):(i + seq_len + predict_ahead), 0])
    return np.array(X), np.array(y)

X_arr, y_arr = create_sequences(scaled_data, seq_len=24, predict_ahead=24)

split = int(0.8 * len(X_arr))
X_train, y_train = X_arr[:split], y_arr[:split]
X_val, y_val = X_arr[split:], y_arr[split:]

X_train_t = torch.tensor(X_train, dtype=torch.float32).transpose(1, 2)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32).transpose(1, 2)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
print(f"Training shape: {X_train_t.shape}")

Training shape: torch.Size([6989, 3, 24])


In [3]:
class CNNLSTMForecast(nn.Module):
    def __init__(self):
        super(CNNLSTMForecast, self).__init__()
        self.conv1d = nn.Conv1d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(input_size=16, hidden_size=32, batch_first=True)
        self.fc = nn.Linear(32, 24)

    def forward(self, x):
        x = self.conv1d(x)
        x = self.relu(x)
        x = x.transpose(1, 2)
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        return self.fc(last_out)

model = CNNLSTMForecast()

In [4]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
batch_size = 64

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for i in range(0, len(X_train_t), batch_size):
        X_batch = X_train_t[i:i+batch_size]
        y_batch = y_train_t[i:i+batch_size]
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_t), y_val_t)
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {epoch_loss/len(X_train_t):.6f} | Val Loss: {val_loss.item():.6f}")

torch.save(model.state_dict(), '../data/adem_forecast.pt')
print("Model exported to adem_forecast.pt")

Epoch [10/50] | Train Loss: 0.000166 | Val Loss: 0.011055
Epoch [20/50] | Train Loss: 0.000156 | Val Loss: 0.010660
Epoch [30/50] | Train Loss: 0.000151 | Val Loss: 0.010505
Epoch [40/50] | Train Loss: 0.000149 | Val Loss: 0.010452
Epoch [50/50] | Train Loss: 0.000146 | Val Loss: 0.010475
Model exported to adem_forecast.pt
